# 04_feature_engineering_liquidity_ict_smc.ipynb

This notebook generates **ICT, SMC, Fair Value Gaps (FVG), Order Blocks (OB)**, and **Liquidity Sweep** features.

### Objectives:
1. Load the market structure dataset.
2. Compute bullish and bearish Fair Value Gaps (FVGs) and their dimensions.
3. Detect bullish and bearish Order Blocks (OBs) based on displacement.
4. Detect liquidity sweeps (e.g. wicks below/above 24h rolling extremes that reclaim the range).
5. Save the computed features to the feature store.

## 1. Environment Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import sys
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")

## 2. Load Dataset

In [ ]:
from utils.config import load_global_config
from utils.io_utils import load_parquet

config = load_global_config(PROJECT_ROOT)
symbol = config.get("data", "symbol", "BTCUSDT")

ms_path = os.path.join(PROJECT_ROOT, "data", "feature_store", f"{symbol}_market_structure.parquet")
df = load_parquet(ms_path)
print(f"Loaded dataset with shape: {df.shape}")

## 3. Compute ICT, SMC & Liquidity Features

In [ ]:
from features.fvg import detect_fvgs
from features.order_blocks import detect_order_blocks
from features.liquidity import detect_liquidity_sweeps

# Compute features
df = detect_fvgs(df)
df = detect_order_blocks(df)
df = detect_liquidity_sweeps(df)

print("ICT / SMC / Liquidity features computed:")
print(df[["bullish_fvg", "bearish_fvg", "bullish_ob", "bearish_ob", "bullish_sweep_24h"]].sum())

## 4. Save Features to Feature Store

In [ ]:
from utils.io_utils import save_parquet

feature_path = os.path.join(PROJECT_ROOT, "data", "feature_store", f"{symbol}_ict_smc_liquidity.parquet")
save_parquet(df, feature_path)
print(f"ICT/SMC/Liquidity features saved to {feature_path}")

## Summary of Generated Features

The following features have been successfully added to the feature store:
1. `bullish_fvg` / `bearish_fvg` (Fair Value Gap indicators)
2. `bullish_ob` / `bearish_ob` (Order Block indicators)
3. `bullish_sweep_24h` / `bearish_sweep_24h` (Liquidity sweeps)

**Next Step**: Run [05_feature_engineering_orderflow_volatility_sessions.ipynb](file:///content/drive/MyDrive/btcusdt_quant_research/notebooks/05_feature_engineering_orderflow_volatility_sessions.ipynb) to engineer CVD, volatility regimes, and session features.

In [ ]:
# Auto-disconnect Colab runtime to save compute units
AUTO_DISCONNECT = False  # Set to True to enable automatic shutdown
if AUTO_DISCONNECT:
    print("Disconnecting Colab runtime...")
    from google.colab import runtime
    runtime.unassign()